# Day 0 — 扩充股票池到30只 (全行业覆盖)

**原有**: 10只 (5只科技)  
**扩充后**: 30只, 覆盖 10 个行业  
**数据源**: Yahoo Finance v8 Chart API (crumb认证)

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import shutil

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"

In [ ]:
# ==========================================
# 第1步: 定义30只股票, 按行业分组
# ==========================================

STOCK_UNIVERSE = {
    "Technology":           ["AAPL", "MSFT", "NVDA", "GOOGL", "CRM"],
    "Financial":            ["JPM", "BAC", "GS", "V"],
    "Healthcare":           ["JNJ", "UNH", "PFE", "ABBV"],
    "Consumer Cyclical":    ["AMZN", "HD", "MCD"],
    "Consumer Defensive":   ["PG", "KO", "WMT"],
    "Energy":               ["XOM", "CVX", "COP"],
    "Industrial":           ["CAT", "BA", "GE"],
    "Communication":        ["META", "NFLX", "DIS"],
    "Materials":            ["LIN", "NEM"],
    "Utilities":            ["NEE", "DUK"],
}

ALL_SYMBOLS = [s for stocks in STOCK_UNIVERSE.values() for s in stocks]
print(f"总股票数: {len(ALL_SYMBOLS)}")
for sector, stocks in STOCK_UNIVERSE.items():
    print(f"  {sector:20s} ({len(stocks)}): {', '.join(stocks)}")

In [ ]:
# ==========================================
# 第2步: Yahoo Finance认证 (获取crumb)
# ==========================================

def get_yahoo_session():
    """创建已认证的Yahoo Finance会话"""
    session = requests.Session()
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    session.headers.update(headers)
    
    # 获取cookie
    session.get("https://fc.yahoo.com/", timeout=10)
    
    # 获取crumb
    r = session.get("https://query2.finance.yahoo.com/v1/test/getcrumb", timeout=10)
    crumb = r.text
    
    return session, crumb

session, crumb = get_yahoo_session()
print(f"认证成功, crumb={crumb}")

In [ ]:
# ==========================================
# 第3步: 下载单只股票历史价格
# ==========================================

def download_prices(symbol, crumb, session, start_date="2015-01-01"):
    """通过Yahoo v8 Chart API下载历史收盘价"""
    
    # Unix时间戳
    period1 = int(pd.Timestamp(start_date).timestamp())
    period2 = int(pd.Timestamp.now().timestamp())
    
    url = f"https://query2.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "crumb": crumb,
    }
    
    r = session.get(url, params=params, timeout=30)
    
    if r.status_code != 200:
        raise Exception(f"HTTP {r.status_code}")
    
    data = r.json()
    result = data.get("chart", {}).get("result", [])
    
    if not result:
        error = data.get("chart", {}).get("error", {})
        raise Exception(f"API Error: {error}")
    
    r0 = result[0]
    timestamps = r0["timestamp"]
    quotes = r0["indicators"]["quote"][0]
    closes = quotes["close"]
    
    # 构建Series
    dates = pd.to_datetime(timestamps, unit="s")
    prices = pd.Series(closes, index=dates, name=symbol)
    
    # 去重 (Yahoo有时返回重复行)
    prices = prices[~prices.index.duplicated(keep="first")]
    
    return prices

In [ ]:
# ==========================================
# 第4步: 批量下载所有股票
# ==========================================

print("批量下载历史价格...")

all_prices = {}
errors = []

for i, sym in enumerate(ALL_SYMBOLS):
    print(f"  [{i+1:2d}/{len(ALL_SYMBOLS)}] {sym:5s} ...", end=" ")
    try:
        prices = download_prices(sym, crumb, session)
        all_prices[sym] = prices
        print(f"OK ({len(prices)} 天)")
    except Exception as e:
        # 刷新crumb重试
        try:
            time.sleep(1)
            session, crumb = get_yahoo_session()
            prices = download_prices(sym, crumb, session)
            all_prices[sym] = prices
            print(f"OK retry ({len(prices)} 天)")
        except Exception as e2:
            print(f"失败: {str(e2)[:60]}")
            errors.append(sym)
    
    time.sleep(0.4)

print(f"\n成功: {len(all_prices)}/{len(ALL_SYMBOLS)}")
if errors:
    print(f"失败: {errors}")

In [ ]:
# ==========================================
# 第5步: 合并为Price矩阵
# ==========================================

df_price = pd.DataFrame(all_prices)
df_price.index.name = "Date"
df_price = df_price.sort_index()

# 只保留 2015-01-01 ~ 2025-12-31
df_price = df_price.loc["2015-01-01":"2025-12-31"]

print(f"价格矩阵: {df_price.shape}")
print(f"日期: {df_price.index[0].date()} ~ {df_price.index[-1].date()}")
print(f"\n各股票缺失天数:")
missing = df_price.isna().sum().sort_values(ascending=False)
for sym, n in missing.items():
    if n > 0:
        print(f"  {sym}: {n} 天 ({n/len(df_price):.1%})")

In [ ]:
# ==========================================
# 第6步: 前向填充缺失 + 丢掉始终NaN的行
# ==========================================

# 前向填充 (同一股票内)
df_price = df_price.fillna(method="ffill")

# 丢弃所有股票都为NaN的行
df_price = df_price.dropna(how="all")

# 对仍NaN的 (某些股票上市晚于2015年) — 丢弃这些股票
still_nan = df_price.isna().sum()
drop_syms = still_nan[still_nan > 0].index.tolist()
if drop_syms:
    print(f"丢弃始终无数据的股票: {drop_syms}")
    df_price = df_price.drop(columns=drop_syms)

print(f"最终价格矩阵: {df_price.shape}")
print(f"股票数: {len(df_price.columns)}")
print(f"日期: {df_price.index[0].date()} ~ {df_price.index[-1].date()}")

In [ ]:
# ==========================================
# 第7步: 备份旧文件 & 保存新文件
# ==========================================

old_file = os.path.join(OUT_DIR, "Prices.csv")
backup_file = os.path.join(OUT_DIR, "Prices_backup_10stocks.csv")

# 备份原文件
if os.path.exists(old_file):
    shutil.copy2(old_file, backup_file)
    print(f"原文件已备份到: Prices_backup_10stocks.csv")

# 保存 (重置index让Date成为列)
df_price.to_csv(old_file, encoding="utf-8-sig", float_format="%.6f")
print(f"新Prices.csv已保存: {df_price.shape}")

# 保存股票-行业映射
sector_rows = []
for sector, stocks in STOCK_UNIVERSE.items():
    for s in stocks:
        if s in df_price.columns:
            sector_rows.append({"symbol": s, "sector": sector})
df_sector = pd.DataFrame(sector_rows)
df_sector.to_csv(os.path.join(OUT_DIR, "stock_sectors.csv"), index=False, encoding="utf-8-sig")
print(f"股票-行业映射已保存: stock_sectors.csv")

print("\n===== Day0 完成! =====")
print("请按顺序重新运行: Day3 → Day4 → Day5 → Day6 → Day7 → Day8 → Day10")